# Base model evaluation
In this notebook, the performance of selected base models will be evaluated on long texts.

**Selected models:**
- Encoder-only:
    - XLM-RoBERTa-large (FacebookAI/xlm-roberta-large) (0.6B parameters)
- Decoder-based:
    - Qwen3-Embedding-0.6B (Qwen/Qwen3-Embedding-0.6B) (0.6B parameters)

In [1]:
import numpy as np

import torch
from datasets import load_dataset
import pandas as pd

import datasets

## Baseline models

In [2]:
from transformers import AutoTokenizer, AutoModel

In [14]:
tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-large")
model = AutoModel.from_pretrained("FacebookAI/xlm-roberta-large")

In [29]:
chunk_size = 4
overlap = 0

inputs = [
    "Some Text 1 e",
    "Some Text 2 i do a little too much"
]

real_chunks_size = chunk_size - 1    # for each chunk eos token will be appened after

number_of_chunks = []    # number of chunks for each text in input
outer_chunked_texts_batch = []    # long batch of all texts chunks

for text in inputs:
    token_ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze()
    start = 0
    chunk_number = 0
    while start < len(token_ids):
        end = start + real_chunks_size
        chunk_tokens = token_ids[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        outer_chunked_texts_batch.append(chunk_text)
        start += real_chunks_size - overlap
        chunk_number += 1
    number_of_chunks.append(chunk_number)

tokenized_outer_batch = tokenizer(
    outer_chunked_texts_batch,
    add_special_tokens=True,
    padding=True,
    truncation=False,
    return_tensors="pt"
)



In [30]:
number_of_chunks

[2, 3]

In [37]:
tokenizer(
    inputs,
    add_special_tokens=True,
    padding="longest",
    truncation=True,
    max_length=4,
    return_tensors="pt")

{'input_ids': tensor([[    0, 31384, 24129,     2],
        [    0, 31384, 24129,     2]]), 'attention_mask': tensor([[1, 1, 1, 1],
        [1, 1, 1, 1]])}

In [57]:
def tokenize_chunking_strategy(tokenizer, inputs, chunk_size):
    real_chunks_size = chunk_size - 1    # for each chunk eos token will be appened after
    
    number_of_chunks = []    # number of chunks for each text in input
    outer_chunked_texts_batch = []    # long batch of all texts chunks
    
    for text in inputs:
        token_ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze()
        start = 0
        chunk_number = 0
        while start < len(token_ids):
            end = start + real_chunks_size
            chunk_tokens = token_ids[start:end]
            chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            outer_chunked_texts_batch.append(chunk_text)
            start += real_chunks_size - overlap
            chunk_number += 1
        number_of_chunks.append(chunk_number)
    
    tokenized_outer_batch = tokenizer(
        outer_chunked_texts_batch,
        add_special_tokens=True,
        padding=True,
        truncation=False,
        return_tensors="pt"
    )
    return (tokenized_outer_batch, number_of_chunks)

def re_group_chunked_outputs(outputs_last_hidden_state, number_of_chunks):
    # outputs shape: [ num_chunks * num_texts, chunk_size, *]
    # converting to [num_texts, num_chunks, chunk_size, *]
    re_grouped = []
    text_starts_i = 0
    for n_chunks in number_of_chunks:
        text_ends_i = text_starts_i + n_chunks 
        re_grouped.append(
            outputs_last_hidden_state[text_starts_i:text_ends_i, :, :]
        )
        text_starts_i = text_ends_i
    return re_grouped

def tokenize_first_startegy(tokenizer, inputs, max_length):
    tokenized = tokenizer(
        inputs,
        add_special_tokens=True,
        padding="longest",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return tokenized

In [44]:
tokenized_outer_batch, number_of_chunks = tokenize_chunking_strategy(tokenizer, inputs, 4)

In [47]:
model = AutoModel.from_pretrained("FacebookAI/xlm-roberta-large", dtype=torch.float32)

In [49]:
outputs = model(**tokenized_outer_batch)

In [55]:
number_of_chunks

[2, 3]

In [59]:
re_group_chunked_outputs(outputs.last_hidden_state, number_of_chunks)

[tensor([[[-0.0660,  0.0426,  0.1319,  ..., -0.0510, -0.0343, -0.0059],
          [ 0.2190,  0.0096,  0.0019,  ...,  0.0626, -0.2342, -0.0073],
          [ 0.1614,  0.0018,  0.1341,  ...,  0.0866, -0.0503, -0.1547],
          [ 0.2631,  0.0336,  0.1128,  ...,  0.0620, -0.3365,  0.0716],
          [ 0.0604, -0.0405,  0.2524,  ...,  0.1969, -0.1120,  0.1695]],
 
         [[-0.0699,  0.0506,  0.0732,  ..., -0.0390, -0.0335, -0.0343],
          [ 0.0555,  0.2153, -0.1640,  ...,  0.3079, -0.2150, -0.0581],
          [ 0.0661,  0.0086,  0.0802,  ...,  0.2098, -0.1145,  0.0700],
          [ 0.0660,  0.0084,  0.0802,  ...,  0.2100, -0.1146,  0.0700],
          [ 0.0660,  0.0084,  0.0802,  ...,  0.2100, -0.1146,  0.0700]]],
        grad_fn=<SliceBackward0>),
 tensor([[[-5.0033e-02,  2.6762e-02,  1.4167e-01,  ..., -5.5675e-02,
           -2.3898e-02, -5.9751e-03],
          [ 2.2087e-01, -2.6393e-03, -1.8705e-02,  ...,  8.1383e-02,
           -2.0647e-01, -1.2051e-02],
          [ 1.3537e-01, -1

In [4]:
class XMLRoBERTa():

    name = "xlm-roberta-large"

    def __init__(self, chunk_size):
        self.chunk_size = chunk_size
        self.model = AutoModel.from_pretrained("FacebookAI/xlm-roberta-large", dtype=torch.float32)
        self.tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-large")
            
        self.model.eval()

        
    def preprocess_batch(inputs, strategy):
        if strategy == "chunking":
            tokenized_inputs =

    def encode(
            self,
            text,
            strategy,
            **kwargs) -> torch.tensor:

        if strategy == "first":
            tokenized_text = self.tokenizer(text,    # whole text
                                            add_special_tokens=True,
                                            padding=False,
                                            truncation=False,
                                            return_tensors="pt")
            # select first chunk_size tokens
            tokenized_first_input_ids = tokenized_text["input_ids"][:, :self.chunk_size]
            attention_mask = tokenized_text["attention_mask"][:, :self.chunk_size]
            with torch.no_grad():
                outputs = self.model(tokenized_first_input_ids, attention_mask)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            return embedding

        if strategy == "last":
            tokenized_text = self.tokenizer(text,    # whole text
                                            add_special_tokens=True,
                                            padding=False,
                                            truncation=False,
                                            return_tensors="pt")
            # select last chunk_size tokens
            tokenized_last_input_ids = tokenized_text["input_ids"][:, -self.chunk_size:]
            attention_mask = tokenized_text["attention_mask"][:, -self.chunk_size:]
            with torch.no_grad():
                outputs = self.model(tokenized_last_input_ids, attention_mask)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            return embedding

        if strategy == "chunking":
            # for text in input make a chunking
            texts_chunked = chunk_text(text, self.tokenizer, self.chunk_size)

            # perform a tokeniztaion of each chunk
            chunk_input_ids = []
            chunk_attention_masks = []

            tokenized = self.tokenizer(
                texts_chunked,    # texts_chunked is effectively a batch of chunks at this point
                padding=True,
                truncation=True,
                max_length=self.chunk_size,
                return_tensors="pt"
            )

            # get embeddings for all chunks
            with torch.no_grad():
                outputs = self.model(**tokenized)
                # mean pooling for each chunk
                embedding = outputs.last_hidden_state.mean(dim=1)
                # mean pooling across chunks:
                embedding = embedding.mean(dim=0)
                return embedding


class Qwen3_Embedding():

    name = "Qwen3-Embedding-0.6B"

    def __init__(self, chunk_size):
        self.chunk_size = chunk_size

        self.tokenizer = AutoTokenizer.from_pretrained(
            "Qwen/Qwen3-Embedding-0.6B",
            padding_side='left')

        self.model = AutoModel.from_pretrained(
            "Qwen/Qwen3-Embedding-0.6B",
            dtype=torch.float32)

        self.model.eval()

    def __get_eos_token_embedding(self, last_hidden_states, attention_mask):
        return last_hidden_states[:, -1]

    def encode(
            self,
            text,
            strategy = "chunking",
            **kwargs) -> torch.tensor:

        if strategy == "first":
            tokenized_text = self.tokenizer(text,    # whole text
                                            add_special_tokens=True,
                                            padding=False,
                                            truncation=False,
                                            return_tensors="pt")
            # select first chunk_size tokens
            tokenized_first_input_ids = tokenized_text["input_ids"][:, :self.chunk_size]
            attention_mask = tokenized_text["attention_mask"][:, :self.chunk_size]

            with torch.no_grad():
                outputs = self.model(tokenized_first_input_ids, attention_mask)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            return embedding

        if strategy == "last":
            tokenized_text = self.tokenizer(text,    # whole text
                                            add_special_tokens=True,
                                            padding=False,
                                            truncation=False,
                                            return_tensors="pt")

            tokenized_first_input_ids = tokenized_text["input_ids"][:, -self.chunk_size:]
            attention_mask = tokenized_text["attention_mask"][:, -self.chunk_size:]

            with torch.no_grad():
                outputs = self.model(tokenized_first_input_ids, attention_mask)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            return embedding

        if strategy == "chunking":
            text_chunked = chunk_text(text, self.tokenizer, self.chunk_size)

            # perform a tokeniztaion of each chunk
            chunk_input_ids = []
            chunk_attention_masks = []

            tokenized = self.tokenizer(
                text_chunked,    # texts_chunked is effectively a batch of chunks at this point
                padding=True,
                truncation=True,
                max_length=self.chunk_size,
                return_tensors="pt"
            )

            # get embeddings for all chunks
            with torch.no_grad():
                outputs = self.model(**tokenized)
                # get an embegging from end of sequence token
                embedding = self.__get_eos_token_embedding(outputs.last_hidden_state, tokenized["attention_mask"])
                # mean pooling for each chunk
                embedding = embedding.mean(dim=0)
                return embedding

## LongEmbed LEMBWikimQARetrieval

In [25]:
from datasets import load_dataset

ds = load_dataset("dwzhu/LongEmbed", name="2wikimqa")
corpus = ds["corpus"]
queries = ds["queries"]
qrels = ds["qrels"]

In [6]:
qwen3_embed = Qwen3_Embedding(512)
xlm_roberta = XMLRoBERTa(512)

### Encoding document with each model with each text-preprocess strategy

In [7]:
# encode each document in a corpus
def encode_documents(corpus, model, strategy):
    document_embeddings = {}
    for doc in corpus:
        embedding = model.encode(doc["text"], strategy)
        document_embeddings[doc["doc_id"]] = embedding.numpy()
    return document_embeddings

# encode each query
def encode_queries(queries, model):
    queries_embeddings = {}
    for query in queries:
        embedding = model.encode(query["text"], strategy="chunking")
        queries_embeddings[query["qid"]] = embedding.numpy()
    return queries_embeddings

In [8]:
# encoding the queries with each mode
# strategy does not really matter in this case since the query will be one chunk long anyway
q3_query_embed = encode_queries(queries, qwen3_embed)
roberta_query_embed = encode_queries(queries, xlm_roberta)

In [15]:
pd.DataFrame(q3_query_embed).to_csv("./query_embed/q3_query_embed.csv")
pd.DataFrame(roberta_query_embed).to_csv("./query_embed/roberta_query_embed.csv")

In [8]:
q3_doc_embed_chunking = encode_documents(corpus, qwen3_embed, strategy="chunking")
roberta_doc_embed_chunking = encode_documents(corpus, xlm_roberta, strategy="chunking")

Token indices sequence length is longer than the specified maximum sequence length for this model (4234 > 512). Running this sequence through the model will result in indexing errors


In [9]:
pd.DataFrame(q3_doc_embed_chunking).to_csv("q3_doc_embed_chunking.csv")
pd.DataFrame(roberta_doc_embed_chunking).to_csv("roberta_doc_embed_chunking.csv")

In [ ]:
q3_doc_embed_first = encode_documents(corpus, qwen3_embed, strategy="first")
roberta_doc_embed_first = encode_documents(corpus, xlm_roberta, strategy="first")

In [ ]:
pd.DataFrame(q3_doc_embed_first).to_csv("q3_doc_embed_first.csv")
pd.DataFrame(roberta_doc_embed_first).to_csv("roberta_doc_embed_first.csv")

In [ ]:
q3_doc_embed_last = encode_documents(corpus, qwen3_embed, strategy="last")
roberta_doc_embed_last = encode_documents(corpus, xlm_roberta, strategy="last")

In [ ]:
pd.DataFrame(q3_doc_embed_last).to_csv("q3_doc_embed_last.csv")
pd.DataFrame(roberta_doc_embed_last).to_csv("roberta_doc_embed_last.csv")

### Calculating evaluation metrics

In [3]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [50]:
def MAP_at_K(documents_embed, queries_embed, qrel, k):
    sum_ap_at_k = 0
    n_queries = 0
    for q in qrel:
        q_id = q["qid"]
        true_doc_id = q["doc_id"]

        q_emebedding = np.array(queries_embed[q_id])
        similarities = []
        for doc_id in documents_embed:
            doc_embedding = np.array(documents_embed[doc_id])
            cos_sim = cosine_similarity(q_emebedding.reshape(1, -1), doc_embedding.reshape(1, -1))[0][0]
            similarities.append({"doc_id": doc_id, "similarity": cos_sim})

        similarities_df = (
            pd.DataFrame(similarities)
            .sort_values(by="similarity", ascending=False)
        )
        top_k = similarities_df["doc_id"].to_list()[:k]

        # assuming that there is only one relevant document for the query
        try:
            true_document_position = top_k.index(true_doc_id) + 1
            ap_at_k = 1 / true_document_position
        except ValueError:
            ap_at_k = 0
        sum_ap_at_k += ap_at_k
        n_queries += 1
    map = sum_ap_at_k / n_queries
    return map

In [44]:
def mean_nDCG_at_k(documents_embed, queries_embed, qrel, k):
    sum_ndcg_at_k = 0
    n_queries = 0

    idcg_at_k = 1
    for i in range(1, 10+1):
        idcg_at_k += 1 / np.log2(i + 2)

    for q in qrel:
        q_id = q["qid"]
        true_doc_id = q["doc_id"]

        q_emebedding = np.array(queries_embed[q_id])
        similarities = []
        for doc_id in documents_embed:
            doc_embedding = np.array(documents_embed[doc_id])
            cos_sim = cosine_similarity(q_emebedding.reshape(1, -1), doc_embedding.reshape(1, -1))[0][0]
            similarities.append({"doc_id": doc_id, "similarity": cos_sim})

        similarities_df = (
            pd.DataFrame(similarities)
            .sort_values(by="similarity", ascending=False)
        )
        top_k = similarities_df["doc_id"].to_list()[:k]

        # assuming that there is only one relevant document for the query
        try:
            true_document_position = top_k.index(true_doc_id) + 1
            dcg_at_k = 1 / np.log2(true_document_position + 1)
        except ValueError:
            dcg_at_k = 0            
        ndcg_at_k = dcg_at_k / idcg_at_k
        sum_ndcg_at_k += ndcg_at_k
        n_queries += 1

    mean_ndcg = sum_ndcg_at_k / n_queries
    return mean_ndcg

In [51]:
roberta_doc_embeddings = pd.read_csv("./doc_embeddings/roberta_doc_embed_chunking.csv").drop(columns="Unnamed: 0").to_dict(orient="list")
q3_doc_embeddings = pd.read_csv("./doc_embeddings/q3_doc_embed_chunking.csv").drop(columns="Unnamed: 0").to_dict(orient="list")

roberta_query_embeddings = pd.read_csv("./query_embeddings/roberta_query_embed.csv").drop(columns="Unnamed: 0").to_dict(orient="list")
q3_query_embeddings = pd.read_csv("./query_embeddings/q3_query_embed.csv").drop(columns="Unnamed: 0").to_dict(orient="list")

In [52]:
print("XLM-RoBERTa-large nDCG@k: ", mean_nDCG_at_k(roberta_doc_embeddings, roberta_query_embeddings, qrels, 10))
print("XLM-RoBERTa-large MAP@k: ", MAP_at_K(roberta_doc_embeddings, roberta_query_embeddings, qrels, 10))

XLM-RoBERTa-large nDCG@k:  0.027452690239719697
XLM-RoBERTa-large MAP@k:  0.10360449735449732


In [53]:
print("Qwen3 Embedding nDCG@k: ", mean_nDCG_at_k(q3_doc_embeddings, q3_query_embeddings, qrels, 10))
print("Qwen3 Embedding MAP@k: ", MAP_at_K(q3_doc_embeddings, q3_query_embeddings, qrels, 10))

Qwen3 Embedding nDCG@k:  0.15436043151119166
Qwen3 Embedding MAP@k:  0.7148743386243387
